# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.### Dataset SourceThe dataset source is provided via the Croissant schema URL below.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data LoadingLoad metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pd# Define the dataset Croissant schema URLcroissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'# Load the dataset metadatadataset = mlc.Dataset(croissant_url)metadata = dataset.metadataprint(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}\nVersion: {getattr(metadata, 'version', 'N/A')}")

## 2. Data OverviewReview available record sets, fields, and their IDs.The Croissant schema organizes tables (record sets), columns, and fields by their unique `@id` references.

In [ ]:
# List available record sets and their @id, name, and fieldsprint("Record sets in the dataset:")if hasattr(metadata, 'record_sets') and metadata.record_sets:    for record_set in metadata.record_sets:        print(f"- {record_set.id}: {getattr(record_set, 'name', 'Unnamed')}")        # List fields for the record set        if hasattr(record_set, 'fields') and record_set.fields:            for field in record_set.fields:                print(f"    - Field @id: {field.id} | Name: {getattr(field, 'name', 'Unnamed')} | DataType: {getattr(field, 'data_type', 'Unknown')}")else:    # For datasets without structured record sets, try to infer from available records    print("No explicit record sets found in the Croissant schema. Attempting to load data via dataset.records()...")    try:        sample_records = list(dataset.records())        if sample_records:            print(f"Number of records sample (default set): {len(sample_records)}")            print(f"Sample record keys: {sample_records[0].keys() if hasattr(sample_records[0],'keys') else sample_records[0]}")    except Exception as e:        print(f"Could not enumerate records: {e}")

## 3. Data ExtractionLoad data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Find available record sets and load all into DataFramesdataframes = {}record_set_ids = []if hasattr(metadata, 'record_sets') and metadata.record_sets:    for record_set in metadata.record_sets:        record_set_ids.append(record_set.id)# If no explicit record_sets, default to loading records without filteringif not record_set_ids:    print("No structured record sets found. Loading all available records as a single DataFrame.")    records = list(dataset.records())    df_default = pd.DataFrame(records)    print("DataFrame columns:", df_default.columns.tolist())    dataframes['default'] = df_default    display(df_default.head())else:    # Load all record sets into DataFrames    for record_set_id in record_set_ids:        records = list(dataset.records(record_set=record_set_id))        df = pd.DataFrame(records)        dataframes[record_set_id] = df        print(f"Loaded DataFrame for record set '@id': {record_set_id}")        print("Columns:", df.columns.tolist())        display(df.head())# For simplicity, select either the only record set, the main one, or the defaultif record_set_ids:    main_record_set_id = record_set_ids[0]  # usually 'cr:recordSet/ProteinAbundanceTable' or similarelse:    main_record_set_id = 'default'# Show fields for the main record setdf = dataframes[main_record_set_id]print(f"Columns in dataframes['{main_record_set_id}']:")print(df.columns.tolist())df.head()

## 4. Exploratory Data Analysis (EDA)Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.Let's analyze a numeric field (for example, 'cr:column/NormalizedAbundance' or 'cr:column/PeptideCount').

In [ ]:
# Inspect to find a numeric columnnumeric_field_candidates = [col for col in df.columns if any(kw in col.lower() for kw in ['abundance', 'count', 'mw', 'weight', 'coverage'])]if numeric_field_candidates:    # Pick the first candidate    numeric_field = numeric_field_candidates[0]    print(f"Selected numeric field: {numeric_field}")else:    # Default to the first column    numeric_field = df.columns[0]# Remove non-numeric entries (if any)df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')# Filter values above a threshold (e.g., mean value)threshold = df[numeric_field].mean()filtered_df = df[df[numeric_field] > threshold]print(f"Filtered records with {numeric_field} > {threshold} (mean value):")print(filtered_df[[numeric_field]].head())# Normalize the numeric field (z-score normalization)filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()print(f"Normalized {numeric_field} for filtered records:")print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())# Group by a categorical field, e.g., if there is a column likely to contain sample or protein namegroup_field_candidates = [col for col in df.columns if any(kw in col.lower() for kw in ['description', 'protein', 'sample', 'type'])]if group_field_candidates:    group_field = group_field_candidates[0]    print(f"Grouping by: {group_field}")    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()    print(f"Grouped mean {numeric_field} by {group_field}:")    print(grouped_df.head())

## 5. VisualizationVisualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as sns# Plot the distribution of the selected numeric fieldplt.figure(figsize=(8, 4))sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)plt.title(f"Distribution of {numeric_field}")plt.xlabel(numeric_field)plt.ylabel('Frequency')plt.show()# If grouped_df exists, show the top 10 groups by mean valueif 'grouped_df' in locals():    plt.figure(figsize=(10, 5))    top_n = grouped_df.sort_values(numeric_field, ascending=False).head(10)    sns.barplot(y=group_field, x=numeric_field, data=top_n)    plt.title(f"Top 10 {group_field} by Mean {numeric_field}")    plt.xlabel(numeric_field)    plt.ylabel(group_field)    plt.show()

## 6. ConclusionWe explored the FAIR² dataset for mass spectrometry analysis of extracellular vesicles. After loading with `mlcroissant`, we:
- Inspected available record sets and fields using `@id` references.
- Loaded the main table into a DataFrame and examined its structure.
- Applied filtering, normalization, and grouping to numeric fields such as protein abundance or counts.
- Visualized value distributions and summarized results by key attributes.

This workflow can be further extended to more advanced analyses or integrated into ML model pipelines.